In [2]:
import torch
import torch.nn as nn
from torchvision.datasets import MNIST
from torchvision.transforms import ToTensor
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

In [3]:
class ContractiveAutoencoder(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(ContractiveAutoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, output_size),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(output_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, input_size),
            nn.Sigmoid()
        )
        self.criterion = nn.MSELoss()
        self.beta = 1e-4

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return encoded, decoded

    def loss_function(self, x, decoded, encoded):
        mse_loss = self.criterion(decoded, x)
        jacobian = torch.autograd.functional.jacobian(lambda x: encoded, x)
        # Compute the Frobenius norm of the Jacobian matrix
        jacobian_norm = torch.norm(jacobian, dim=(2, 3))
        contraction_loss = torch.mean(torch.sum(jacobian_norm ** 2, dim=1))
        total_loss = mse_loss + (self.beta * contraction_loss)
        return total_loss

    def generate_reconstruction(self, labels, condition):
        # Generate latent space representation based on condition
        with torch.no_grad():
            latent = torch.randn((labels.size(0), self.hidden_size)).to(labels.device)
            latent[:, condition] = self.encoder(labels)

        # Reconstruct the data from the modified latent space
        with torch.no_grad():
            reconstructed = self.decoder(latent)

        return reconstructed

In [4]:
# Set the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Hyperparameters
input_size = 784  # MNIST images are 28x28 pixels
hidden_size = 128
output_size = 64
learning_rate = 0.001
num_epochs = 10
batch_size = 128

In [ ]:
# Load MNIST dataset
train_dataset = MNIST(root="./data", train=True, transform=ToTensor(), download=True)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [ ]:
# Initialize the autoencoder
autoencoder = ContractiveAutoencoder(input_size, hidden_size, output_size).to(device)

# Define the optimizer
optimizer = torch.optim.Adam(autoencoder.parameters(), lr=learning_rate)


In [ ]:
# Training loop
total_steps = len(train_dataloader)
for epoch in range(num_epochs):
    for i, (images, labels) in enumerate(train_dataloader):
        # Move images and labels to the device
        images = images.to(device)
        labels = labels.to(device)
        inputs = images.view(images.size(0), -1)

        # Forward pass
        encoded, decoded = autoencoder(inputs)

        # Compute the loss
        loss = autoencoder.loss_function(inputs, decoded, encoded)

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (i + 1) % 50 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{total_steps}], Loss: {loss.item():.4f}")

In [ ]:
# Generate reconstructions based on conditions or labels
condition = 5  # Modify this to the desired condition or label
sample_labels = torch.randint(low=0, high=10, size=(5,))  # Generate 5 random labels
reconstructions = autoencoder.generate_reconstruction(sample_labels.to(device), condition).cpu().view(-1, 28, 28)

# Display the reconstructions
fig, axes = plt.subplots(nrows=1, ncols=5, figsize=(10, 4))

for i in range(5):
    axes[i].imshow(reconstructions[i], cmap='gray')
    axes[i].axis('off')
    axes[i].set_title(f'Reconstructed\nLabel: {sample_labels[i]}')

plt.tight_layout()
plt.show()